In [ ]:
# Carga las credenciales desde config/.env (nunca las escribas en el notebook).
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent / "src"))

from quant_engine.config import get_settings

settings = get_settings()


In [ ]:
import yfinance as yf
datos = yf.download("AAPL", start ="2023-01-01", end="2023-12-31")
#print(datos) #muestra todos los datos pero con puntos suspensivos si es mucha data
#datos.head() #muestra una tabla pero solo los primero 5 filas
#print(type(datos)) #devuelve el tipo usado en este caso pandas
#print(datos.columns) #devuelve la informacion de las columnas dependiendo del ticket
print(datos.index) #devuelve la matriz de la 

Alpha Vantage

In [ ]:
import requests
url = f"https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=IBM&interval=5min&apikey={settings.alpha_vantage_api_key}"
r = requests.get(url)
data = r.json()

print(data)

Twelve Data

In [ ]:
from twelvedata import TDClient

# Initialize client - apikey parameter is requiered
td = TDClient(apikey=settings.twelve_data_api_key)

# Construct the necessary time series
ts = td.time_series(
    symbol="FXAIX", #TODO DEPENDE DEL TICKET ESTA ES LA ESTRUCTURA BASE
    interval="1min",
    outputsize=10,
    timezone="America/New_York",
)

# Returns pandas.DataFrame
ts.as_pandas()

Finnhub

In [ ]:
import finnhub

finnhub_client = finnhub.Client(api_key=settings.finnhub_api_key)



In [ ]:
import sys

sys.modules.pop("websocket", None)
import websocket

def on_message(ws, message):
    print(message)

def on_error(ws, error):
    print(error)

def on_close(ws, close_status_code, close_msg):
    print("### closed ###")

def on_open(ws):
    ws.send('{"type":"subscribe","symbol":"AAPL"}')
    ws.send('{"type":"subscribe","symbol":"AMZN"}')
    ws.send('{"type":"subscribe","symbol":"BINANCE:BTCUSDT"}')
    ws.send('{"type":"subscribe","symbol":"IC MARKETS:1"}')

if __name__ == "__main__":
    ws = websocket.WebSocketApp(f"wss://ws.finnhub.io?token={settings.finnhub_api_key}",
                                 on_message=on_message,
                                 on_error=on_error,
                                 on_close=on_close)
    ws.on_open = on_open
    ws.run_forever()

Marketstack

In [ ]:
import requests

url = f"https://api.ipstack.com/?access_key={settings.ipstack_api_key}"

querystring = {"symbols":"AAPL"}
response = requests.get(url, params=querystring)
print(response.json())

In [ ]:
import http.client
conn = http.client.HTTPSConnection("api.marketstack.com")
conn.request("GET", f"/v1/eod?access_key={settings.marketstack_api_key}&symbols=AAPL")
res = conn.getresponse()
data = res.read()
print(data)

Tiingo

In [ ]:
import requests

headers = {
        'Content-Type': 'application/json'
        }
requestResponse = requests.get(f"https://api.tiingo.com/api/test?token={settings.tiingo_api_key}",
                                    headers=headers)
print(requestResponse.json())
                        

In [ ]:
import requests

headers = {
        'Content-Type': 'application/json',
        'Authorization' : f'Token {settings.tiingo_api_key}'
        }
requestResponse = requests.get("https://api.tiingo.com/api/test/",
                                    headers=headers)
print(requestResponse.json())

In [ ]:
import requests
headers = {
    'Content-Type': 'application/json'
}
requestResponse = requests.get(f"https://api.tiingo.com/tiingo/daily/aapl/prices?startDate=2019-01-02&token={settings.tiingo_api_key}", headers=headers)
print(requestResponse.json())

In [ ]:
import yfinance as yf
import openpyxl
Universo_yf = pd.read_excel("../config/SP500_yfinance.xlsx")
Tickets_yf = Universo_yf["Ticket"].tolist() #convertira en lista la columna del excel
Datos_yf = yf.download(Tickets_yf, start ="2026-08-20", end="2026-08-21")
Datos_yf.head()

In [ ]:
def cierres_sector(Universo_yf,sector, start, end):
    Tk_filtrado = Universo_yf [Universo_yf["Sector"]==sector]["Ticket"].tolist()
    datos = yf.download(Tk_filtrado, start = start, end=end)
    return datos[["Close", "Open"]] #Doble corchete debido a que son columnas

In [ ]:
finanzas = cierres_sector(Universo_yf,"Financials","2023-02-01","2023-09-01") 
finanzas.head()

In [ ]:
import yfinance as yf
datos =yf.download("AAPL", start="2023-01-01", end = "2026-12-01")
datos.head()

In [ ]:
from quant_engine.ingestion.dt_yfinance import cargar_universo, cierre_sect_yf

Universo = cargar_universo("../config/SP500_yfinance.xlsx")
Finanzas = cierre_sect_yf(Universo, "Financials", "2023-02-01", "2023-09-01")
Finanzas.head()
